# Data Preprocessing

This notebook prepares the sepsis dataset for machine learning.

## Objectives

The preprocessing pipeline will:

1. Load and validate the dataset.
2. Perform basic data cleaning.
3. Split patients into training, validation, and test sets.
4. Preserve patient-level separation to prevent data leakage.
5. Apply patient-wise forward filling to clinical measurements.
6. Convert clinical measurements into reference-range deviation scores.
7. Compress the 34 clinical variables into five clinically inspired domain scores.
8. Retain demographic and temporal information separately.
9. Remove the original high-dimensional clinical variables.
10. Prepare final feature matrices for modeling.

### Important methodological notes

- `Patient_ID` is retained in processed datasets for patient-level tracking and grouping.
- `Patient_ID` is never used as a model feature.
- Imputation and feature-engineering decisions are learned/applied without using validation or test labels.
- Clinical reference ranges are used as feature-engineering heuristics and are not diagnostic thresholds.
- The resulting clinical domain scores are project-specific and are not clinically validated.

___

In [68]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [69]:
df = pd.read_csv("../data/raw/raw_Dataset.csv")

In [70]:
df = df.drop(columns=df.columns[0])

___
**Basic validation**

In [71]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing Patient_ID:", df["Patient_ID"].isna().sum())
print("Missing SepsisLabel:", df["SepsisLabel"].isna().sum())
print("Unique patients:", df["Patient_ID"].nunique())

Shape: (1552210, 43)
Duplicate rows: 0
Missing Patient_ID: 0
Missing SepsisLabel: 0
Unique patients: 40336


___
**Feature group**

In [72]:
clinical_features = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "EtCO2", "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2",
    "SaO2", "AST", "BUN", "Alkalinephos", "Calcium", "Chloride",
    "Creatinine", "Bilirubin_direct", "Glucose", "Lactate",
    "Magnesium", "Phosphate", "Potassium", "Bilirubin_total",
    "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen",
    "Platelets"
]

demographic_features = [
    "Age",
    "Gender"
]

categorical_features = [
    "Unit1",
    "Unit2"
]

temporal_features = [
    "Hour",
    "HospAdmTime",
    "ICULOS"
]

target = "SepsisLabel"
id_column = "Patient_ID"

In [73]:
assert len(clinical_features) == 34
assert target in df.columns
assert id_column in df.columns

print("Clinical features:", len(clinical_features))
print("Feature groups validated.")

Clinical features: 34
Feature groups validated.


___
**Patient level split**

In [74]:
patients = df["Patient_ID"].unique()

train_val_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

train_patients, val_patients = train_test_split(
    train_val_patients,
    test_size=0.125,
    random_state=42
)

In [75]:
train_df = df[df["Patient_ID"].isin(train_patients)].copy()
val_df = df[df["Patient_ID"].isin(val_patients)].copy()
test_df = df[df["Patient_ID"].isin(test_patients)].copy()

___
**Verification**

In [76]:
assert set(train_df["Patient_ID"]).isdisjoint(val_df["Patient_ID"])
assert set(train_df["Patient_ID"]).isdisjoint(test_df["Patient_ID"])
assert set(val_df["Patient_ID"]).isdisjoint(test_df["Patient_ID"])

print("Patient-level separation verified.")

Patient-level separation verified.


___
**Processed datasets**

In [77]:
train_processed = train_df.copy()
val_processed = val_df.copy()
test_processed = test_df.copy()

___
**Sorting Values for Patients**

In [78]:
train_processed = train_processed.sort_values(
    ["Patient_ID", "Hour"]
)

val_processed = val_processed.sort_values(
    ["Patient_ID", "Hour"]
)

test_processed = test_processed.sort_values(
    ["Patient_ID", "Hour"]
)

In [79]:
train_processed = train_processed.reset_index(drop=True)
val_processed = val_processed.reset_index(drop=True)
test_processed = test_processed.reset_index(drop=True)

___
**Patient wise forward fill**

In [80]:
forward_fill_features = [
    feature for feature in clinical_features
    if feature != "Age"
]

In [81]:
for data in [train_processed, val_processed, test_processed]:
    data[forward_fill_features] = (
        data.groupby("Patient_ID")[forward_fill_features]
        .ffill()
    )

___
**defining clinical reference range**

In [82]:
clinical_reference_ranges = {
    # Hemodynamic
    "HR": (60, 100),
    "SBP": (90, 140),
    "MAP": (65, 100),
    "DBP": (60, 80),

    # Respiratory
    "Resp": (12, 20),
    "O2Sat": (94, 100),
    "Temp": (36.0, 37.5),
    "EtCO2": (35, 45),
    "FiO2": (0.21, 0.60),
    "pH": (7.35, 7.45),
    "PaCO2": (35, 45),
    "SaO2": (94, 100),

    # Acid-base / metabolic
    "BaseExcess": (-2, 2),
    "HCO3": (22, 26),
    "Lactate": (0.5, 2.0),
    "Glucose": (70, 140),

    # Renal / electrolytes
    "BUN": (7, 20),
    "Creatinine": (0.6, 1.3),
    "Calcium": (8.5, 10.5),
    "Chloride": (98, 107),
    "Magnesium": (1.7, 2.2),
    "Phosphate": (2.5, 4.5),
    "Potassium": (3.5, 5.0),

    # Hepatic
    "AST": (10, 40),
    "Alkalinephos": (44, 147),
    "Bilirubin_total": (0.1, 1.2),
    "Bilirubin_direct": (0.0, 0.3),

    # Hematologic / coagulation
    "WBC": (4, 11),
    "Platelets": (150, 450),
    "Hgb": (12, 17),
    "Hct": (36, 50),
    "PTT": (25, 35),
    "Fibrinogen": (200, 400),

    # Cardiac marker
    "TroponinI": (0, 0.04)
}

In [83]:
assert set(clinical_features) == set(clinical_reference_ranges)

print("All 34 clinical features have reference ranges.")

All 34 clinical features have reference ranges.


___
**Calculating deviations**

In [84]:
def calculate_deviation(value, lower, upper):
    if pd.isna(value):
        return np.nan

    reference_width = upper - lower

    if lower <= value <= upper:
        return 0.0
    elif value < lower:
        return (lower - value) / reference_width
    else:
        return (value - upper) / reference_width

In [85]:
clinical_deviation_train = pd.DataFrame(index=train_processed.index)
clinical_deviation_val = pd.DataFrame(index=val_processed.index)
clinical_deviation_test = pd.DataFrame(index=test_processed.index)

In [87]:
for feature in clinical_features:
    lower, upper = clinical_reference_ranges[feature]

    clinical_deviation_train[feature] = train_processed[feature].apply(
        lambda x: calculate_deviation(x, lower, upper)
    )

    clinical_deviation_val[feature] = val_processed[feature].apply(
        lambda x: calculate_deviation(x, lower, upper)
    )

    clinical_deviation_test[feature] = test_processed[feature].apply(
        lambda x: calculate_deviation(x, lower, upper)
    )

In [88]:
print("Train deviation shape:", clinical_deviation_train.shape)
print("Validation deviation shape:", clinical_deviation_val.shape)
print("Test deviation shape:", clinical_deviation_test.shape)

Train deviation shape: (1086991, 34)
Validation deviation shape: (155226, 34)
Test deviation shape: (309993, 34)


___
**compressing the deviating values**

In [89]:
clinical_deviation_train = np.log1p(clinical_deviation_train)
clinical_deviation_val = np.log1p(clinical_deviation_val)
clinical_deviation_test = np.log1p(clinical_deviation_test)

___

## Clinical Domain Reduction

The 34 clinical variables are grouped into five clinically inspired domains:

1. Hemodynamic / Cardiovascular
2. Respiratory
3. Renal / Metabolic / Perfusion
4. Inflammatory / Hematological
5. Hepatic / Coagulation

Each domain will produce a single weighted abnormality score.

These domain scores are project-specific feature-engineering representations and are not validated clinical scores.

___

In [90]:
clinical_domains = {
    "Hemodynamic": [
        "HR",
        "SBP",
        "MAP",
        "DBP"
    ],

    "Respiratory": [
        "Resp",
        "O2Sat",
        "Temp",
        "EtCO2",
        "FiO2",
        "pH",
        "PaCO2",
        "SaO2"
    ],

    "Renal_Metabolic": [
        "BaseExcess",
        "HCO3",
        "BUN",
        "Creatinine",
        "Calcium",
        "Chloride",
        "Magnesium",
        "Phosphate",
        "Potassium",
        "Glucose",
        "Lactate"
    ],

    "Inflammatory_Hematological": [
        "WBC",
        "Platelets",
        "Hgb",
        "Hct",
        "Fibrinogen"
    ],

    "Hepatic_Coagulation": [
        "AST",
        "Alkalinephos",
        "Bilirubin_total",
        "Bilirubin_direct",
        "PTT",
        "TroponinI"
    ]
}

___
**Validating the groups**

In [91]:
all_domain_features = [
    feature
    for features in clinical_domains.values()
    for feature in features
]

assert len(all_domain_features) == 34
assert len(set(all_domain_features)) == 34
assert set(all_domain_features) == set(clinical_features)

print("All 34 clinical features assigned exactly once.")

All 34 clinical features assigned exactly once.


___
**Creating feature evidence weights**

In [92]:
effect_sizes = {}

for feature in clinical_features:
    non_sepsis = train_processed.loc[
        train_processed["SepsisLabel"] == 0,
        feature
    ].dropna()

    sepsis = train_processed.loc[
        train_processed["SepsisLabel"] == 1,
        feature
    ].dropna()

    pooled_std = np.sqrt(
        (non_sepsis.var() + sepsis.var()) / 2
    )

    if pooled_std == 0 or pd.isna(pooled_std):
        effect_sizes[feature] = 0.0
    else:
        effect_sizes[feature] = abs(
            sepsis.mean() - non_sepsis.mean()
        ) / pooled_std

In [93]:
effect_df = pd.DataFrame.from_dict(
    effect_sizes,
    orient="index",
    columns=["EffectSize"]
)

effect_df.index.name = "Feature"

___
**Normalizing the evidences**

In [94]:
effect_df["EvidenceWeight"] = (
    effect_df["EffectSize"] /
    effect_df["EffectSize"].max()
)

In [95]:
effect_df.sort_values(
    "EvidenceWeight",
    ascending=False
).head(10)

,EffectSize,EvidenceWeight
Feature,,
HR,0.342275,1.000000
Temp,0.324666,0.948553
BUN,0.291852,0.852684
Resp,0.284214,0.830368
Fibrinogen,0.275896,0.806064
WBC,0.229493,0.670494
Bilirubin_direct,0.214939,0.627972
Hgb,0.204680,0.598000
Hct,0.189447,0.553495


___
**Adding Clinical importance priors**

In [96]:
clinical_importance = {
    # Hemodynamic
    "HR": 1.0,
    "SBP": 1.0,
    "MAP": 1.0,
    "DBP": 0.8,

    # Respiratory
    "Resp": 1.0,
    "O2Sat": 1.0,
    "Temp": 1.0,
    "EtCO2": 0.8,
    "FiO2": 0.8,
    "pH": 1.0,
    "PaCO2": 0.9,
    "SaO2": 0.9,

    # Renal / metabolic
    "BaseExcess": 0.8,
    "HCO3": 0.8,
    "BUN": 0.9,
    "Creatinine": 1.0,
    "Calcium": 0.6,
    "Chloride": 0.6,
    "Magnesium": 0.6,
    "Phosphate": 0.6,
    "Potassium": 0.8,
    "Glucose": 0.7,
    "Lactate": 1.0,

    # Hematological
    "WBC": 1.0,
    "Platelets": 0.8,
    "Hgb": 0.7,
    "Hct": 0.7,
    "Fibrinogen": 0.8,

    # Hepatic / coagulation
    "AST": 0.7,
    "Alkalinephos": 0.5,
    "Bilirubin_total": 0.8,
    "Bilirubin_direct": 0.6,
    "PTT": 0.8,
    "TroponinI": 0.7
}

In [97]:
assert set(clinical_importance) == set(clinical_features)

print("Clinical importance weights validated.")

Clinical importance weights validated.


___

**Availability**

In [98]:
availability = (
    train_processed[clinical_features]
    .notna()
    .mean()
)

availability_weight = np.sqrt(availability)

___
#### Combine the three components

Our feature weight is:

we'll use soft weighting:

$$ W = C \times E_{\text{soft}} \times A_{\text{soft}} $$

where:

$$ E_{\text{soft}} = 0.25 + 0.75E $$

and

$$ A_{\text{soft}} = 0.50 + 0.50{\sqrt{A}} $$
	​



where:

* C = clinical importance
* E = training-data evidence
* A = measurement availability

In [99]:
weights = pd.DataFrame(index=clinical_features)

weights["EffectSize"] = effect_df["EffectSize"]

weights["EvidenceWeight"] = (
    effect_df["EvidenceWeight"] * 0.75 + 0.25
)

weights["ClinicalImportance"] = pd.Series(
    clinical_importance
)

weights["Availability"] = availability

weights["AvailabilityWeight"] = (
    0.50 + 0.50 * np.sqrt(availability)
)

In [100]:
weights["FinalWeight"] = (
    weights["ClinicalImportance"]
    * weights["EvidenceWeight"]
    * weights["AvailabilityWeight"]
)

In [101]:
for domain, features in clinical_domains.items():
    domain_total = weights.loc[features, "FinalWeight"].sum()

    weights.loc[features, "DomainWeight"] = (
        weights.loc[features, "FinalWeight"] / domain_total
    )

In [102]:
weights[
    [
        "EffectSize",
        "ClinicalImportance",
        "Availability",
        "EvidenceWeight",
        "AvailabilityWeight",
        "FinalWeight",
        "DomainWeight"
    ]
].sort_values(
    "FinalWeight",
    ascending=False
)

,EffectSize,ClinicalImportance,Availability,EvidenceWeight,AvailabilityWeight,FinalWeight,DomainWeight
HR,0.342275,1.0,0.977306,1.000000,0.994294,0.994294,0.397080
Temp,0.324666,1.0,0.926436,0.961415,0.981258,0.943396,0.276112
Resp,0.284214,1.0,0.969693,0.872776,0.992365,0.866112,0.253492
BUN,0.291852,0.9,0.795805,0.889513,0.946040,0.757363,0.222611
WBC,0.229493,1.0,0.776059,0.752870,0.940471,0.708053,0.311468
MAP,0.159692,1.0,0.972296,0.599920,0.993025,0.595736,0.237912
SBP,0.107182,1.0,0.961199,0.484859,0.990204,0.480109,0.191736
Hgb,0.204680,0.7,0.790789,0.698500,0.944632,0.461878,0.203177
Creatinine,0.106105,1.0,0.787167,0.482499,0.943612,0.455292,0.133823
Fibrinogen,0.275896,0.8,0.108538,0.854548,0.664726,0.454432,0.199902


In [103]:
for domain, features in clinical_domains.items():
    print(
        domain,
        "→",
        round(weights.loc[features, "DomainWeight"].sum(), 4)
    )

Hemodynamic → 1.0
Respiratory → 1.0
Renal_Metabolic → 1.0
Inflammatory_Hematological → 1.0
Hepatic_Coagulation → 1.0


In [104]:
def calculate_domain_score(deviation_df, weights_df, features):
    weighted_sum = pd.Series(0.0, index=deviation_df.index)
    available_weight = pd.Series(0.0, index=deviation_df.index)

    for feature in features:
        observed = deviation_df[feature].notna()

        weighted_sum.loc[observed] += (
            deviation_df.loc[observed, feature]
            * weights_df.loc[feature, "DomainWeight"]
        )

        available_weight.loc[observed] += (
            weights_df.loc[feature, "DomainWeight"]
        )

    domain_score = weighted_sum / available_weight

    # No observed variables in the domain → keep as missing
    domain_score[available_weight == 0] = np.nan

    return domain_score

In [105]:
domain_scores_train = pd.DataFrame(index=train_processed.index)
domain_scores_val = pd.DataFrame(index=val_processed.index)
domain_scores_test = pd.DataFrame(index=test_processed.index)

for domain, features in clinical_domains.items():

    domain_scores_train[domain] = calculate_domain_score(
        clinical_deviation_train,
        weights,
        features
    )

    domain_scores_val[domain] = calculate_domain_score(
        clinical_deviation_val,
        weights,
        features
    )

    domain_scores_test[domain] = calculate_domain_score(
        clinical_deviation_test,
        weights,
        features
    )

In [106]:
print("Train:", domain_scores_train.shape)
print("Validation:", domain_scores_val.shape)
print("Test:", domain_scores_test.shape)

print("\nDomain columns:")
print(domain_scores_train.columns.tolist())

Train: (1086991, 5)
Validation: (155226, 5)
Test: (309993, 5)

Domain columns:
['Hemodynamic', 'Respiratory', 'Renal_Metabolic', 'Inflammatory_Hematological', 'Hepatic_Coagulation']


In [107]:
domain_scores_train.head()

,Hemodynamic,Respiratory,Renal_Metabolic,Inflammatory_Hematological,Hepatic_Coagulation
0,NaN,NaN,NaN,NaN,NaN
1,0.000000,0.000000,NaN,NaN,NaN
2,0.000000,0.159891,NaN,NaN,NaN
3,0.000000,0.603688,1.871802,NaN,NaN
4,0.034736,0.506634,1.871802,NaN,NaN


In [108]:
domain_scores_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Hemodynamic,1062506.0,0.071374,0.088588,0.0,0.000000,0.045461,0.104180,1.209244
Respiratory,1062626.0,0.094108,0.117655,0.0,0.000000,0.050961,0.148073,1.897120
Renal_Metabolic,954605.0,0.209380,0.250656,0.0,0.040042,0.110180,0.283884,2.261763
Inflammatory_Hematological,880148.0,0.221645,0.167447,0.0,0.096100,0.207163,0.312265,3.451800
Hepatic_Coagulation,688765.0,0.490250,0.974315,0.0,0.000000,0.093487,0.566039,9.305651


In [109]:
domain_missingness = (
    domain_scores_train
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

domain_missingness

Hepatic_Coagulation           36.635630
Inflammatory_Hematological    19.028952
Renal_Metabolic               12.179126
Hemodynamic                    2.252549
Respiratory                    2.241509
dtype: float64

In [110]:
domain_scores_train.isna().sum()

Hemodynamic                    24485
Respiratory                    24365
Renal_Metabolic               132386
Inflammatory_Hematological    206843
Hepatic_Coagulation           398226
dtype: int64

In [111]:
for column in domain_scores_train.columns:
    observed = domain_scores_train[column].dropna()

    print(f"\n{column}")
    print(f"Observed count : {len(observed):,}")
    print(f"Missing count  : {domain_scores_train[column].isna().sum():,}")
    print(f"Missing %      : {domain_scores_train[column].isna().mean() * 100:.2f}%")
    print(f"Median         : {observed.median():.6f}")
    print(f"Mean           : {observed.mean():.6f}")


Hemodynamic
Observed count : 1,062,506
Missing count  : 24,485
Missing %      : 2.25%
Median         : 0.045461
Mean           : 0.071374

Respiratory
Observed count : 1,062,626
Missing count  : 24,365
Missing %      : 2.24%
Median         : 0.050961
Mean           : 0.094108

Renal_Metabolic
Observed count : 954,605
Missing count  : 132,386
Missing %      : 12.18%
Median         : 0.110180
Mean           : 0.209380

Inflammatory_Hematological
Observed count : 880,148
Missing count  : 206,843
Missing %      : 19.03%
Median         : 0.207163
Mean           : 0.221645

Hepatic_Coagulation
Observed count : 688,765
Missing count  : 398,226
Missing %      : 36.64%
Median         : 0.093487
Mean           : 0.490250


In [112]:
domain_scores_train.corr()

,Hemodynamic,Respiratory,Renal_Metabolic,Inflammatory_Hematological,Hepatic_Coagulation
Hemodynamic,1.000000,0.122751,0.074730,0.057290,0.003768
Respiratory,0.122751,1.000000,0.106999,0.113887,0.002973
Renal_Metabolic,0.074730,0.106999,1.000000,0.174078,0.083487
Inflammatory_Hematological,0.057290,0.113887,0.174078,1.000000,-0.038820
Hepatic_Coagulation,0.003768,0.002973,0.083487,-0.038820,1.000000


In [113]:
domain_scores_train[
    [
        "Hemodynamic",
        "Respiratory",
        "Renal_Metabolic",
        "Inflammatory_Hematological",
        "Hepatic_Coagulation"
    ]
].corr()["Hepatic_Coagulation"].sort_values(ascending=False)

Hepatic_Coagulation           1.000000
Renal_Metabolic               0.083487
Hemodynamic                   0.003768
Respiratory                   0.002973
Inflammatory_Hematological   -0.038820
Name: Hepatic_Coagulation, dtype: float64

## Clinical Feature Weighting and Domain Score Construction

To reduce the dimensionality of the clinical variables while preserving clinically meaningful information, the original clinical features were grouped into five physiological domains:

- **Hemodynamic**
- **Respiratory**
- **Renal/Metabolic**
- **Inflammatory/Hematological**
- **Hepatic/Coagulation**

Before constructing these domain scores, several analyses were performed, including:

- Effect-size analysis between sepsis and non-sepsis observations
- Clinical-importance assessment
- Feature availability/missingness analysis
- Evidence-based feature weighting
- Domain-level weighting

These analyses were used to determine how strongly each clinical variable should contribute to its respective domain score. The resulting weights were therefore part of the **feature-engineering process**, rather than additional model features.

The clinical variables were converted into deviation-based scores relative to clinically meaningful reference ranges. These deviations were transformed and weighted, and then aggregated within each physiological domain. This reduced the original set of clinical variables to **five domain-level features** while retaining information about abnormal physiological patterns.

### Why the weighting and analysis tables are not included in the final dataset

The intermediate weighting tables, effect-size tables, clinical-importance values, availability weights, and domain weights are retained for **documentation, interpretation, and reproducibility**. They explain how the five domain scores were constructed.

They are **not added as separate predictors** to the final modelling dataset because doing so would unnecessarily increase dimensionality and could introduce redundant information.

Thus, the final clinical representation consists only of:

1. `Hemodynamic`
2. `Respiratory`
3. `Renal_Metabolic`
4. `Inflammatory_Hematological`
5. `Hepatic_Coagulation`

The intermediate weighting information is therefore kept as part of the preprocessing/feature-engineering methodology but is **left out of the final feature matrix**.

### Handling of Missing Domain Scores

The domain scores were deliberately **not median-imputed at this stage**.

The remaining missingness was:

- Hemodynamic: ~2.25%
- Respiratory: ~2.24%
- Renal/Metabolic: ~12.18%
- Inflammatory/Hematological: ~19.03%
- Hepatic/Coagulation: ~36.64%

Because the domain scores are not strongly correlated with one another, particularly the heavily missing Hepatic/Coagulation domain, iterative or KNN imputation may introduce information that is not actually supported by the observed data.

Median imputation was also avoided because it would place a large number of observations at exactly the same value and could distort the distribution, especially for the highly skewed Hepatic/Coagulation domain.

Therefore, the missing domain scores are currently **preserved as missing values**. The appropriate treatment will be determined according to the missing-value capabilities of the final machine-learning model.

This maintains the principle that:

> **Missing clinical evidence should not automatically be interpreted as normal clinical evidence.**

___
___

___
___

In [114]:
domain_features = [
    "Hemodynamic",
    "Respiratory",
    "Renal_Metabolic",
    "Inflammatory_Hematological",
    "Hepatic_Coagulation"
]

temporal_features = [
    "Hour",
    "ICULOS",
    "HospAdmTime"
]

categorical_features = [
    "Gender",
    "Unit1",
    "Unit2"
]

final_features = (
    domain_features
    + temporal_features
    + categorical_features
)

print("Number of final features:", len(final_features))
print(final_features)

Number of final features: 11
['Hemodynamic', 'Respiratory', 'Renal_Metabolic', 'Inflammatory_Hematological', 'Hepatic_Coagulation', 'Hour', 'ICULOS', 'HospAdmTime', 'Gender', 'Unit1', 'Unit2']


___

In [115]:
# Create final feature matrices

X_train = pd.concat(
    [
        domain_scores_train,
        train_processed[temporal_features + categorical_features]
    ],
    axis=1
)

X_val = pd.concat(
    [
        domain_scores_val,
        val_processed[temporal_features + categorical_features]
    ],
    axis=1
)

X_test = pd.concat(
    [
        domain_scores_test,
        test_processed[temporal_features + categorical_features]
    ],
    axis=1
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1086991, 11)
X_val shape: (155226, 11)
X_test shape: (309993, 11)


In [116]:
print("Train alignment:",
      X_train.index.equals(train_processed.index))

print("Validation alignment:",
      X_val.index.equals(val_processed.index))

print("Test alignment:",
      X_test.index.equals(test_processed.index))

Train alignment: True
Validation alignment: True
Test alignment: True


In [117]:
y_train = train_processed["SepsisLabel"].copy()
y_val = val_processed["SepsisLabel"].copy()
y_test = test_processed["SepsisLabel"].copy()

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

y_train shape: (1086991,)
y_val shape: (155226,)
y_test shape: (309993,)


In [118]:
print("Training target percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation target percentage:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTest target percentage:")
print(y_test.value_counts(normalize=True) * 100)

Training target percentage:
SepsisLabel
0    98.2274
1     1.7726
Name: proportion, dtype: float64

Validation target percentage:
SepsisLabel
0    98.21164
1     1.78836
Name: proportion, dtype: float64

Test target percentage:
SepsisLabel
0    98.105764
1     1.894236
Name: proportion, dtype: float64


In [119]:
print("\nMissing percentage in X_train:")
print((X_train.isna().mean() * 100).round(2))

print("\nMissing percentage in X_val:")
print((X_val.isna().mean() * 100).round(2))

print("\nMissing percentage in X_test:")
print((X_test.isna().mean() * 100).round(2))


Missing percentage in X_train:
Hemodynamic                    2.25
Respiratory                    2.24
Renal_Metabolic               12.18
Inflammatory_Hematological    19.03
Hepatic_Coagulation           36.64
Hour                           0.00
ICULOS                         0.00
HospAdmTime                    0.00
Gender                         0.00
Unit1                         39.42
Unit2                         39.42
dtype: float64

Missing percentage in X_val:
Hemodynamic                    2.30
Respiratory                    2.29
Renal_Metabolic               12.46
Inflammatory_Hematological    19.08
Hepatic_Coagulation           36.20
Hour                           0.00
ICULOS                         0.00
HospAdmTime                    0.00
Gender                         0.00
Unit1                         39.49
Unit2                         39.49
dtype: float64

Missing percentage in X_test:
Hemodynamic                    2.31
Respiratory                    2.27
Renal_Metabol

In [120]:
print("Gender values:")
print(X_train["Gender"].value_counts(dropna=False))

print("\nUnit1 values:")
print(X_train["Unit1"].value_counts(dropna=False))

print("\nUnit2 values:")
print(X_train["Unit2"].value_counts(dropna=False))

Gender values:
Gender
1    604280
0    482711
Name: count, dtype: int64

Unit1 values:
Unit1
NaN    428504
0.0    331503
1.0    326984
Name: count, dtype: int64

Unit2 values:
Unit2
NaN    428504
1.0    331503
0.0    326984
Name: count, dtype: int64


In [121]:
def create_icu_unit(df):
    return np.where(
        df["Unit1"].notna(),
        df["Unit1"],
        np.nan
    )

In [122]:
X_train["ICU_Unit"] = X_train["Unit1"]
X_val["ICU_Unit"] = X_val["Unit1"]
X_test["ICU_Unit"] = X_test["Unit1"]

In [123]:
X_train = X_train.drop(columns=["Unit1", "Unit2"])
X_val = X_val.drop(columns=["Unit1", "Unit2"])
X_test = X_test.drop(columns=["Unit1", "Unit2"])

In [124]:
print("X_train shape:", X_train.shape)
print("Final features:")
print(X_train.columns.tolist())

X_train shape: (1086991, 10)
Final features:
['Hemodynamic', 'Respiratory', 'Renal_Metabolic', 'Inflammatory_Hematological', 'Hepatic_Coagulation', 'Hour', 'ICULOS', 'HospAdmTime', 'Gender', 'ICU_Unit']


In [125]:
for data in [X_train, X_val, X_test]:
    data["ICU_Unit"] = data["ICU_Unit"].fillna(-1)

In [126]:
print("X_train ICU_Unit:")
print(X_train["ICU_Unit"].value_counts())

print("\nX_val ICU_Unit:")
print(X_val["ICU_Unit"].value_counts())

print("\nX_test ICU_Unit:")
print(X_test["ICU_Unit"].value_counts())

X_train ICU_Unit:
ICU_Unit
-1.0    428504
 0.0    331503
 1.0    326984
Name: count, dtype: int64

X_val ICU_Unit:
ICU_Unit
-1.0    61297
 0.0    47008
 1.0    46921
Name: count, dtype: int64

X_test ICU_Unit:
ICU_Unit
-1.0    122159
 0.0     94838
 1.0     92996
Name: count, dtype: int64


In [127]:
print("Remaining missing values in X_train:")
print(X_train.isna().sum())

print("\nRemaining missing percentage in X_train:")
print((X_train.isna().mean() * 100).round(2))

Remaining missing values in X_train:
Hemodynamic                    24485
Respiratory                    24365
Renal_Metabolic               132386
Inflammatory_Hematological    206843
Hepatic_Coagulation           398226
Hour                               0
ICULOS                             0
HospAdmTime                        0
Gender                             0
ICU_Unit                           0
dtype: int64

Remaining missing percentage in X_train:
Hemodynamic                    2.25
Respiratory                    2.24
Renal_Metabolic               12.18
Inflammatory_Hematological    19.03
Hepatic_Coagulation           36.64
Hour                           0.00
ICULOS                         0.00
HospAdmTime                    0.00
Gender                         0.00
ICU_Unit                       0.00
dtype: float64


In [128]:
domain_features = [
    "Hemodynamic",
    "Respiratory",
    "Renal_Metabolic",
    "Inflammatory_Hematological",
    "Hepatic_Coagulation"
]

print("Rows where each domain score is missing:\n")

for feature in domain_features:
    print(
        feature,
        ":",
        X_train[feature].isna().sum(),
        "rows"
    )

Rows where each domain score is missing:

Hemodynamic : 24485 rows
Respiratory : 24365 rows
Renal_Metabolic : 132386 rows
Inflammatory_Hematological : 206843 rows
Hepatic_Coagulation : 398226 rows


In [129]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nNumber of features:", X_train.shape[1])

print("\nFinal features:")
print(X_train.columns.tolist())

print("\nAny completely empty rows:")
print("Train:", X_train.isna().all(axis=1).sum())
print("Validation:", X_val.isna().all(axis=1).sum())
print("Test:", X_test.isna().all(axis=1).sum())

X_train: (1086991, 10)
X_val: (155226, 10)
X_test: (309993, 10)

Number of features: 10

Final features:
['Hemodynamic', 'Respiratory', 'Renal_Metabolic', 'Inflammatory_Hematological', 'Hepatic_Coagulation', 'Hour', 'ICULOS', 'HospAdmTime', 'Gender', 'ICU_Unit']

Any completely empty rows:
Train: 0
Validation: 0
Test: 0


In [130]:
train_final = X_train.copy()
val_final = X_val.copy()
test_final = X_test.copy()

train_final["SepsisLabel"] = y_train.values
val_final["SepsisLabel"] = y_val.values
test_final["SepsisLabel"] = y_test.values

In [131]:
train_final["Patient_ID"] = train_processed["Patient_ID"].values
val_final["Patient_ID"] = val_processed["Patient_ID"].values
test_final["Patient_ID"] = test_processed["Patient_ID"].values

In [132]:
print("Train:", train_final.shape)
print("Validation:", val_final.shape)
print("Test:", test_final.shape)

print("\nColumns:")
print(train_final.columns.tolist())

Train: (1086991, 12)
Validation: (155226, 12)
Test: (309993, 12)

Columns:
['Hemodynamic', 'Respiratory', 'Renal_Metabolic', 'Inflammatory_Hematological', 'Hepatic_Coagulation', 'Hour', 'ICULOS', 'HospAdmTime', 'Gender', 'ICU_Unit', 'SepsisLabel', 'Patient_ID']


In [133]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [134]:
train_final.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

val_final.to_csv(
    "../data/processed/val_processed.csv",
    index=False
)

test_final.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

___
___

## Final Feature Representation and Missing-Value Strategy

After preprocessing and clinical feature engineering, the original 34 clinical variables were reduced to five physiological domain scores:

- Hemodynamic
- Respiratory
- Renal/Metabolic
- Inflammatory/Hematological
- Hepatic/Coagulation

These domain scores were combined with selected temporal and contextual variables:

- Hour
- ICULOS
- HospAdmTime
- Gender
- ICU_Unit

This resulted in a final feature representation of 10 predictors.

### Dimensionality Reduction

The original 34 clinical variables were not directly passed to the final feature matrix. Instead, they were transformed into five domain-level scores using clinical deviation, log1p compression, and domain-specific feature weighting.

The `Unit1` and `Unit2` variables were found to be perfectly complementary in the dataset. Therefore, they were represented by a single `ICU_Unit` feature to avoid redundant predictors.

`Patient_ID` was excluded from the final feature matrix because it is an identifier rather than a clinical predictor.

`SepsisLabel` was retained separately as the prediction target.

### Missing-Value Strategy

Missing values in the clinical domain scores were deliberately preserved.

The remaining missingness was:

- Hemodynamic: approximately 2.25%
- Respiratory: approximately 2.24%
- Renal/Metabolic: approximately 12.18%
- Inflammatory/Hematological: approximately 19.03%
- Hepatic/Coagulation: approximately 36.64%

No observation had all five clinical domain scores missing.

Median imputation was not applied because replacing missing clinical evidence with a population median could introduce artificial values and distort the distributions of the domain scores, particularly for the highly missing Hepatic/Coagulation domain.

*The missing domain values will therefore be handled according to the missing-value capabilities of the selected machine-learning model.*

### Final Feature Set

The final modelling representation contains 10 predictors:

1. Hemodynamic
2. Respiratory
3. Renal_Metabolic
4. Inflammatory_Hematological
5. Hepatic_Coagulation
6. Hour
7. ICULOS
8. HospAdmTime
9. Gender
10. ICU_Unit

___
___

In [135]:
# import joblib

# joblib.dump(
#     weights,
#     "../models/clinical_domain_weights.pkl"
# )

# print("Clinical domain weights saved successfully.")